# Lekcija 08 - Vzorec večagentnega oblikovanja


## Namestitev


In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

%pip install agent-framework azure-ai-projects azure-identity --quiet

import os
import asyncio

from agent_framework import AgentResponseUpdate, WorkflowBuilder
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Zakaj sistemi z več agenti?

Dejavnosti iz resničnega sveta, kot je načrtovanje potovanja, vključujejo veliko različnih vrst strokovnega znanja — logistiko, lokalno znanje, proračun in še več. En sam agent, ki poskuša upravljati vse, hitro postane neučinkovit.

Sistemi z več agenti to rešujejo s **specializacijo**: vsak agent se osredotoči na eno področje strokovnosti in ustvari bolj kakovostne rezultate kot splošnež. Prav tako izboljšajo **razširljivost** — lahko dodate nove agente (npr. strokovnjaka za lete, kritika restavracij) brez prepisovanja obstoječega delovnega procesa. Agenti sodelujejo skozi strukturiran potek dela, pri čemer kontekst predajajo iz enega v drugega.


## Ustvarjanje specializiranih agentov


In [ ]:
planner_agent = await provider.create_agent(
    name="TravelPlanner",
    instructions="You are a travel planning specialist. Create detailed trip itineraries based on the traveler's preferences. Include daily schedules, must-see attractions, and logistical tips.",
)

concierge_agent = await provider.create_agent(
    name="TravelConcierge",
    instructions="You are a travel concierge who reviews and enhances trip plans. Review the plan for completeness, add local insider tips, suggest restaurants, and identify potential issues. Provide your feedback in a constructive format.",
)

## Sestavljanje zaporednega poteka dela

`WorkflowBuilder` vam omogoča povezovanje agentov v usmerjeni graf. Tukaj ustvarimo enostaven dvostopenjski potek: **TravelPlanner** pripravi načrt poti, nato pa ga **TravelConcierge** pregleda in izboljša.


In [ ]:
workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .build()

last_author = None
events = workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## Dodajanje več agentov v potek dela

Ena največjih prednosti vzorca z več agenti je, kako enostavno ga je razširiti. Spodaj dodamo agenta **BudgetReviewer**, ki preveri načrt glede na proračun popotnika, označi predmete, ki bi lahko povzročili preseganje stroškov, in predlaga alternative za varčevanje z denarjem. Potek dela zdaj zaporedno izvaja tri agente:

```
TravelPlanner → TravelConcierge → BudgetReviewer
```


In [ ]:
budget_agent = await provider.create_agent(
    name="BudgetReviewer",
    instructions="You are a budget-conscious travel advisor. Review the proposed trip plan and concierge enhancements against the traveler's stated budget. Estimate costs for flights, hotels, meals, and activities. Flag anything that risks exceeding the budget and suggest cost-saving alternatives while preserving the trip's quality.",
)

extended_workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .add_edge(concierge_agent, budget_agent) \
    .build()

last_author = None
events = extended_workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## Povzetek

V tej lekciji ste se naučili:

1. **Ustvariti specializirane agente** — vsak s poudarjeno vlogo (načrtovanje, concierge, pregled proračuna).
2. **Povezati agente v zaporedno delovno tok** z uporabo `WorkflowBuilder` in `add_edge`.
3. **Pretakati izhod** iz cevovoda z več agenti, pri čemer sledite, kateri agent govori.
4. **Razširiti delovni tok** z dodajanjem novih agentov v verigo, brez spreminjanja obstoječih.

Vzorec oblikovanja z več agenti ohranja vsak agent preprost, hkrati pa proizvaja bogatejše, bolj temeljito pregledane rezultate, kot jih lahko doseže kateri koli posamezen agent.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Omejitev odgovornosti**:  
Ta dokument je bil preveden z uporabo AI prevajalske storitve [Co-op Translator](https://github.com/Azure/co-op-translator). Čeprav si prizadevamo za natančnost, vas opozarjamo, da lahko avtomatizirani prevodi vsebujejo napake ali netočnosti. Izvirni dokument v njegovem izvirnem jeziku velja za verodostojen vir. Za ključne informacije priporočamo strokovni človeški prevod. Nismo odgovorni za kakršna koli nesporazumevanja ali napačne interpretacije, ki izhajajo iz uporabe tega prevoda.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
